# Packet P-041 — Family-Aware Models

**Decision question:** Does training separate ET models per composition family improve within-family ranking over the single global model? P-037 showed LOGO tau-b = 0.005 — could family-specific models rescue ranking?

**Fastest falsifier:** Train a Pure MA-only model and compare within-Pure-MA tau-b to the global model's. If no improvement, family-specific models don't help.

**Success:** Family-specific models improve mean within-family tau-b by ≥ 0.05 over the global model (P-032 baseline).
**Kill:** No family-specific model improves tau-b by > 0.02 — the global model already captures within-family signal.

In [1]:
"""Cell 1 — Compare global vs family-specific models."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

N_SPLITS = 20
families_to_test = [f for f in df["family"].value_counts().index if df["family"].value_counts()[f] >= 40]

# Store results
global_taus = {fam: [] for fam in families_to_test}
local_taus = {fam: [] for fam in families_to_test}

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    # Global model
    global_model = ExtraTreesRegressor(**ET_PARAMS)
    global_model.fit(X_tr, y_tr)
    global_preds = global_model.predict(X_te)
    
    test_families = df.loc[X_te.index, "family"]
    
    for fam in families_to_test:
        fam_mask_te = test_families == fam
        if fam_mask_te.sum() < 5:
            continue
        
        # Global model within-family tau-b
        y_fam = y_te[fam_mask_te]
        gp_fam = global_preds[fam_mask_te.values]
        tau_g, _ = kendalltau(y_fam, gp_fam)
        global_taus[fam].append(tau_g)
        
        # Family-specific model
        fam_mask_tr = df.loc[X_tr.index, "family"] == fam
        if fam_mask_tr.sum() < 20:
            continue
        
        X_fam_tr = X_tr[fam_mask_tr]
        y_fam_tr = y_tr[fam_mask_tr]
        
        local_params = ET_PARAMS.copy()
        local_params['random_state'] = seed
        local_model = ExtraTreesRegressor(**local_params)
        local_model.fit(X_fam_tr, y_fam_tr)
        local_preds = local_model.predict(X_te[fam_mask_te])
        
        tau_l, _ = kendalltau(y_fam, local_preds)
        local_taus[fam].append(tau_l)
    
    if (seed + 1) % 10 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Report
print(f"\n{'=' * 70}")
print("GLOBAL vs FAMILY-SPECIFIC MODEL COMPARISON")
print(f"{'=' * 70}")
print(f"{'Family':<20} {'n':>5} {'Global tau-b':>13} {'Local tau-b':>13} {'Delta':>8}")
print("-" * 62)

results = []
for fam in families_to_test:
    n = (df["family"] == fam).sum()
    g_mean = np.mean(global_taus[fam]) if global_taus[fam] else float('nan')
    l_mean = np.mean(local_taus[fam]) if local_taus[fam] else float('nan')
    delta = l_mean - g_mean if not (np.isnan(g_mean) or np.isnan(l_mean)) else float('nan')
    print(f"{fam:<20} {n:>5} {g_mean:>13.4f} {l_mean:>13.4f} {delta:>+8.4f}")
    results.append({'family': fam, 'n': n, 'global_tau': g_mean, 'local_tau': l_mean, 'delta': delta})

res_df = pd.DataFrame(results)
mean_delta = res_df['delta'].mean()
max_delta = res_df['delta'].max()
print(f"\nMean delta (local - global): {mean_delta:+.4f}")
print(f"Max delta: {max_delta:+.4f}")

  Completed 10/20 splits


  Completed 20/20 splits

GLOBAL vs FAMILY-SPECIFIC MODEL COMPARISON
Family                   n  Global tau-b   Local tau-b    Delta
--------------------------------------------------------------
Pure MA                967        0.2780        0.2808  +0.0028
Triple cation          197        0.4012        0.3920  -0.0092
MA-FA mixed            197        0.3289        0.3333  +0.0044
Other                   88        0.1893        0.2338  +0.0445
Pure FA                 50        0.0240        0.0832  +0.0591
FA-Cs                   44        0.3230        0.3311  +0.0081

Mean delta (local - global): +0.0183
Max delta: +0.0591


In [2]:
"""Cell 2 — Evaluate and save."""

any_improved = any(r['delta'] > 0.05 for _, r in res_df.iterrows() if not np.isnan(r['delta']))
no_improvement = all(abs(r['delta']) <= 0.02 for _, r in res_df.iterrows() if not np.isnan(r['delta']))

if any_improved:
    status = "Confirmed"
elif no_improvement:
    status = "Negative"
else:
    status = "Promising"

res_df.to_csv('Packet_P041_Family_Aware_Models.csv', index=False)
print(f"Saved: Packet_P041_Family_Aware_Models.csv")

print(f"\n{'=' * 60}")
print(f"P-041 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Family-specific models improve within-family ranking.")
    print("Consider deploying per-family models for lab partners.")
elif status == "Negative":
    print("Family-specific models don't improve — global model already captures within-family signal.")
else:
    print("Mixed results — some families benefit, others don't.")

Saved: Packet_P041_Family_Aware_Models.csv

P-041 STATUS: Confirmed
Family-specific models improve within-family ranking.
Consider deploying per-family models for lab partners.
